# Master Large-Model Visualization, Results, Targets, and PRT Workflow

This is the canonical integration notebook for the new `myflopy` result-review stack. By default it builds and runs a **10,000-cell, four-layer, six-period DISV/Voronoi model** with CHD, GHB, DRN, UZF, SFR, two lakes, MVR routing, canonical observations, standalone Matplotlib sliders, Plotly animations, and MF6 PRT.

The default `full` profile is intentionally substantial. Automated tests set `SIMPLE_MODFLOW_MASTER_PROFILE=validation`, which preserves the same package topology on a smaller grid.

In [ ]:
from __future__ import annotations

import os
from datetime import datetime
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import LineString

import myflopy as mf
from myflopy import plot
from myflopy.modflow.mf6 import plot_model_cross_section

sys.path.append(str(Path.home() / "Python/Projects/myflopy/examples/mf6"))


from visualization_prt_master_support import (
    MasterExampleConfig,
    build_transient_model,
    representative_cells,
)

PROFILE = os.environ.get("SIMPLE_MODFLOW_MASTER_PROFILE", "full").lower()
RUN_PRT = os.environ.get("SIMPLE_MODFLOW_MASTER_RUN_PRT", "1") == "1"
EXPORT_3D = os.environ.get("SIMPLE_MODFLOW_MASTER_EXPORT_3D", "1") == "1"
EXPORT_ALL_SLIDERS = os.environ.get("SIMPLE_MODFLOW_MASTER_EXPORT_ALL_SLIDERS", "1") == "1"
EXPORT_PLOTLY_SECTION = os.environ.get("SIMPLE_MODFLOW_MASTER_EXPORT_PLOTLY_SECTION", "1") == "1"
FRAME_STRIDE = int(os.environ.get("SIMPLE_MODFLOW_MASTER_FRAME_STRIDE", "1"))
MAX_FRAMES = os.environ.get("SIMPLE_MODFLOW_MASTER_MAX_FRAMES")
MAX_FRAMES = None if MAX_FRAMES is None else int(MAX_FRAMES)
RESUME_EXPORTS = os.environ.get("SIMPLE_MODFLOW_MASTER_RESUME_EXPORTS", "1") == "1"
config = MasterExampleConfig.validation() if PROFILE == "validation" else MasterExampleConfig()

repo_root = Path(mf.__file__).resolve().parents[2]
run_id = os.environ.get("SIMPLE_MODFLOW_MASTER_RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S"))
artifact_root = repo_root / "examples" / "mf6" / "artifacts" / f"master_visualization_prt_{PROFILE}_{run_id}"
artifact_root.mkdir(parents=True, exist_ok=True)
html_dir = artifact_root / "html"
html_dir.mkdir(exist_ok=True)

print(f"Profile: {PROFILE}")
print(f"Plan-view cells: {config.ncpl:,}")
print(f"Layers: {config.nlay}; total GWF cells: {config.ncpl * config.nlay:,}")
print(f"Stress periods: {config.nper}")

## 1. Build the package-rich model

The support builder creates all geometry in code, writes temporary GeoPackages for the surface-water intersections, and attaches canonical targets before the run. The resulting object is a normal `SimulationBase` model.

In [ ]:
model = build_transient_model(artifact_root / "gwf", config=config, name="master_complex")
cells = representative_cells(config)

assert model.gwf.modelgrid.grid_type == "vertex"
assert model.gwf.modelgrid.ncpl == config.ncpl
assert model.gwf.modelgrid.nlay == config.nlay
if PROFILE == "full":
    assert model.gwf.modelgrid.ncpl >= 10_000

pd.DataFrame(
    {
        "metric": ["ncpl", "nlay", "total cells", "nper", "package count"],
        "value": [config.ncpl, config.nlay, config.ncpl * config.nlay, config.nper, len(model.gwf.package_names)],
    }
)

In [ ]:
required_packages = {"disv", "npf", "sto", "chd", "ghb", "drn", "lak", "sfr", "mvr", "uzf", "oc"}
actual_packages = set(model.gwf.package_names)
assert required_packages <= actual_packages
sorted(actual_packages)

## 2. Inspect canonical groups, targets, and MF6 observations

The same target objects define observation locations, attach valid MF6 observation records, and later compare simulated results. SFR stage and flow definitions are combined into one SFR `OBS6` child because MF6 permits only one `OBS6` reference per package.

In [ ]:
target_summary = pd.DataFrame(
    {
        "target": ["heads", "lake_stage", "sfr_stage", "sfr_flow", "drn_flow"],
        "type": [
            type(model.targets.heads.targets).__name__,
            type(model.targets.lake_stage.targets).__name__,
            type(model.targets.sfr_stage.targets).__name__,
            type(model.targets.sfr_flow.targets).__name__,
            type(model.targets.drn_flow.targets).__name__,
        ],
    }
)
target_summary

In [ ]:
region_names = ["all_lakes", "all_streams", "uzf_active"]
region_counts = {name: len(model.get_region_cells(name)) for name in region_names}
assert all(region_counts.values())
region_counts

## 3. Run MF6 and verify complex-package outputs

In [ ]:
success, report = model.run_simulation()
assert success, "\n".join(report[-30:])

kstpkpers = list(model.gwf.output.head().get_kstpkper())
head_frames = [np.squeeze(model.gwf.output.head().get_data(kstpkper=value), axis=1) for value in kstpkpers]
assert len(kstpkpers) == config.nper * config.steps_per_period
assert head_frames[-1].shape == (config.nlay, config.ncpl)
print(f"Saved head frames: {len(head_frames)}; final shape: {head_frames[-1].shape}")

In [ ]:
output_files = {
    "heads": model.model_output_folder_path / f"{model.name}.hds",
    "budget": model.model_output_folder_path / f"{model.name}.cbc",
    "lake budget csv": model.model_output_folder_path / f"{model.name}_lake_budget.csv",
    "UZF budget csv": model.model_output_folder_path / f"{model.name}_uzf_budget.csv",
    "MVR budget csv": model.model_output_folder_path / f"{model.name}.mvr.csv",
    "head observations": model.model_output_folder_path / "master_heads.csv",
    "lake observations": model.model_output_folder_path / "master_lakes.csv",
    "SFR stage observations": model.model_output_folder_path / "master_sfr_stage.csv",
    "SFR flow observations": model.model_output_folder_path / "master_sfr_flow.csv",
    "DRN observations": model.model_output_folder_path / "master_drn.csv",
}
assert all(path.exists() for path in output_files.values())
pd.DataFrame({"output": output_files.keys(), "size_bytes": [path.stat().st_size for path in output_files.values()]})

## 4. Static Matplotlib result views

These use FloPy-compatible map and cross-section renderers and are the same rendering path used by the standalone slider exporter.

In [ ]:
map_style = mf.ModelMapStyle(figsize=(11, 8), cmap="terrain", contour_levels=12, dpi=100)
fig, ax = mf.plot_model_head_map(model, head_frames[-1], layer=0, style=map_style, title="Final heads, layer 1")
plt.show()

In [ ]:
section_line = LineString([(config.cell_size * 0.5, config.cell_size * config.nrow * 0.5), (config.cell_size * (config.ncol - 0.5), config.cell_size * config.nrow * 0.5)])
fig, ax = plot_model_cross_section(model, section_line, head_data=head_frames[-1], show_legend=False, title="Final multilayer head cross section")
plt.show()

## 5. Standalone Matplotlib HTML sliders

`embed_frames=True` creates one portable HTML file. For this large model, `embed_frames=False` is preferred: it writes PNG frames beside a small HTML controller and avoids assembling every base64 image in memory. External-frame exports are resumable, report progress, reject concurrent writers, and support `frame_stride` / `max_frames` for bounded review runs.

In [ ]:
embedded_preview = mf.export_head_map_slider_html(
    model,
    html_dir / "head_map_embedded_preview.html",
    head_frames=head_frames[:2],
    labels=[str(value) for value in kstpkpers[:2]],
    layer=0,
    style=mf.ModelMapStyle(show_contours=False, dpi=75),
    embed_frames=True,
)
assert embedded_preview.embedded_frames and embedded_preview.path.exists()
embedded_preview

In [ ]:
head_slider = mf.export_head_map_slider_html(
    model,
    html_dir / "head_map_all_times.html",
    layer=0,
    style=mf.ModelMapStyle(show_contours=True, contour_levels=10, dpi=85),
    embed_frames=False,
    frame_stride=FRAME_STRIDE,
    max_frames=MAX_FRAMES,
    resume=RESUME_EXPORTS,
    progress=True,
)
sliders = [head_slider]
if EXPORT_ALL_SLIDERS:
    sliders.extend([
        mf.export_head_layer_mosaic_slider_html(
            model,
            html_dir / "head_mosaic_all_layers.html",
            layers=list(range(config.nlay)),
            ncols=2,
            style=mf.ModelMapStyle(show_contours=False, dpi=70, figsize=(6, 5)),
            embed_frames=False,
            frame_stride=FRAME_STRIDE,
            max_frames=MAX_FRAMES,
            resume=RESUME_EXPORTS,
            progress=True,
        ),
        mf.export_cross_section_slider_html(
            model,
            section_line,
            html_dir / "head_cross_section_all_times.html",
            show_legend=False,
            dpi=80,
            embed_frames=False,
            frame_stride=FRAME_STRIDE,
            max_frames=MAX_FRAMES,
            resume=RESUME_EXPORTS,
            progress=True,
        ),
    ])

for slider in sliders:
    assert slider.path.exists()
    expected_frames = len(kstpkpers[::FRAME_STRIDE]) if MAX_FRAMES is None else min(MAX_FRAMES, len(kstpkpers[::FRAME_STRIDE]))
    assert slider.frame_count == expected_frames
    assert slider.frame_directory is not None
    assert len(list(slider.frame_directory.glob("*.png"))) >= expected_frames

[slider.path for slider in sliders]

## 6. Plotly animations

`plot.animate(frames)` flips through pictures you build -- so the frame
selection is explicit, which matters here: an animation costs one rendered map
per frame. `.html(path)` writes a standalone page; for a choropleth it ships the
cell geometry once and restyles only the values, rather than re-embedding the
whole grid in every frame.

In [ ]:
animated_periods = kstpkpers[::FRAME_STRIDE][:MAX_FRAMES]
plotly_map = plot.animate([
    (str(period), model.plot.map(kstpkper=period, layer=0,
                                 show_layer_elevs=False, hover_heads=True))
    for period in animated_periods
])
plotly_map.html(html_dir / "plotly_head_map.html")
assert len(plotly_map.fig.frames) == expected_frames
plotly_map.show()

In [ ]:
plotly_section = None
if EXPORT_PLOTLY_SECTION:
    plotly_section = plot.animate([
        (str(period), model.plot.section(
            kstpkper=period,
            cells=cells["cross_section"],
            layer=list(range(config.nlay)),
            interpolate=False,
            use_rbf=False,
        ))
        for period in animated_periods
    ])
    plotly_section.html(html_dir / "plotly_head_cross_section.html")
    assert len(plotly_section.fig.frames) == expected_frames
plotly_section

## 7. Canonical target and package-result review

Targets remain independent of PEST. They can inspect the real MF6 observation files or package results produced above.

In [ ]:
simulated_targets = {
    "heads": model.targets.heads.targets.simulated_heads(model),
    "lake_stage": model.targets.lake_stage.targets.simulated_series(model),
    "sfr_stage": model.targets.sfr_stage.targets.simulated_series(model),
    "sfr_flow": model.targets.sfr_flow.targets.simulated_series(model),
    "drn_flow": model.targets.drn_flow.targets.simulated_series(model),
}
assert all(not frame.empty for frame in simulated_targets.values())
{name: frame.shape for name, frame in simulated_targets.items()}

In [ ]:
package_review = {
    "lake_stage_rows": len(model.packages.lak.results.stage.get()),
    "sfr_exchange_rows": len(model.packages.sfr.results.q.get()),
    "uzf_ifno_count": len(model.outputs.uzf.ifno_to_cellid),
    "budget_terms": model.bud().types,
}
package_review

## 8. MF6 PRT particle tracking

PRT is a separate MF6 simulation that copies the GWF DISV grid and timing, then consumes completed GWF head and budget outputs through FMI. MP3DU remains available through `model.particle_tracking.mp3du(...)`, but PRT is the preferred integrated engine.

In [ ]:
release_points = mf.PRTReleasePoints.from_cells(
    model,
    cells=cells["releases"],
    layer=0,
    local_z=0.5,
)
prt_project = model.particle_tracking.prt(
    workspace=artifact_root / "prt",
    release_points=release_points,
    porosity=0.25,
    stop_at_weak_sink=False,
)
prt_project.write()
assert (prt_project.workspace / "mfsim.nam").exists()
prt_project.workspace

In [ ]:
prt_result = prt_project.run(write=False, silent=True) if RUN_PRT else None
if prt_result is not None:
    assert prt_result.success
    # `pathlines` is a view: .get() is the normalized record table.
    tracks = prt_result.pathlines.get()
    assert not tracks.empty
    display(tracks.head())
    display(prt_result.terminal_points.head())

In [ ]:
if prt_result is not None:
    # Interactive: one polyline per particle over the final-period water table.
    prt_result.pathlines.map(layer=0, title='PRT pathlines')

    reopened = mf.open_prt_run(model, prt_project.workspace)
    assert len(reopened.pathlines.get()) == len(prt_result.pathlines.get())

    if EXPORT_3D:
        particle_html = reopened.export_3d_html(
            html_dir / "prt_particle_scene.html",
            vertical_exaggeration=5.0,
        )
        assert particle_html.exists()
        print(particle_html)

## 9. Final integration audit

This final cell makes the notebook act like an integration test. If it completes, the model, complex packages, canonical observations, result readers, Matplotlib sliders, Plotly animations, and optional PRT/PyVista export all worked together.

In [ ]:
audit = {
    "profile": PROFILE,
    "ncpl": config.ncpl,
    "nlay": config.nlay,
    "total_gwf_cells": config.ncpl * config.nlay,
    "saved_head_frames": len(kstpkpers),
    "packages": sorted(required_packages),
    "standalone_html_exports": len(list(html_dir.glob("*.html"))),
    "prt_ran": prt_result is not None,
    "prt_rows": 0 if prt_result is None else len(prt_result.track_records),
}
minimum_exports = 7 if EXPORT_ALL_SLIDERS and EXPORT_PLOTLY_SECTION and EXPORT_3D else 3
assert audit["standalone_html_exports"] >= minimum_exports
pd.Series(audit, name="master workflow audit")